# 03 - QLoRA Training

Three adapters, one per expert. `train_qlora.py --role all` trains them
sequentially, one model resident at a time.

## The consistency requirement

**Every adapter must be trained with the same budget.** If one expert gets more
epochs, longer sequences or more examples than another, then a topology
difference is partly *'which graph position holds the best-trained expert'* - a
confound stacked on the model-position one the paper already discloses, and one
entirely of our own making.

So changing any hyperparameter means **retraining all three**, not topping up the
ones not yet built. `train_qlora.py` enforces this: it resumes by default, but
**aborts** if an existing adapter was trained on a different budget.

## 3.1 Final hyperparameters, and the two revisions that got here

| parameter | plan | final | why |
|---|---|---|---|
| epochs | 3 | **1** | held-out loss bottomed at epoch 1-2 on all three; epoch 3 made every adapter worse |
| learning rate | 2e-4 | **5e-5** | answer length collapsed 161->68 words at 2e-4 |
| max_seq_length | 1024 | **1024** | 512 truncated 83% of NewsQA targets (see nb 02) |
| LoRA r / alpha / dropout | 16 / 32 / 0.05 | unchanged | |
| effective batch | 8 | 8 | batch 1 x 8 grad-accum |
| optimiser | paged AdamW 8-bit | unchanged | the QLoRA paper's choice |
| quantisation | 4-bit NF4 + double quant | unchanged | |

Cost: **~19-44 min per adapter, ~91 min for all three.**

Targets are rendered through **each model's own chat template**, so an adapter is
trained in the format it is served in. Training on a different scaffold than
deployment is a common and avoidable reason a fine-tune appears not to have
worked - and it bit us later at *validation* time (see nb 04).

In [ ]:
# Train all three. Defaults are the final values above, so no flags needed.
!.venv\Scripts\python.exe -u finetune\train_qlora.py --role all

# Resumes by default: a completed adapter is skipped, so an interrupted run
# continues instead of redoing ~50 minutes per expert. Use --force to overwrite.

## 3.2 Evaluate on step intervals, not per epoch

With a single epoch, per-epoch evaluation yields **one data point** - not a
curve, and unable to show whether the run saturated or started degrading. That
was the diagnostic that mattered most on the failed attempt. `SFTConfig` now
uses `eval_strategy='steps'` with four evaluations per run at any epoch count.

In [ ]:
# Results of the final run. All three monotonic, converging, no degradation.
import json
for role in ('legal', 'news', 'general_qa'):
    m = json.load(open(f'adapters/{role}/training_meta.json', encoding='utf-8'))
    curve = [round(p['eval_loss'], 3) for p in m['eval_history']]
    print(f"{role:11s} trainable={m['trainable_params']:>11,} "
          f"({100 * m['trainable_params'] / m['total_params']:.2f}%)  "
          f"train_loss={m['final_train_loss']:.3f}  {m['duration_s']/60:5.1f} min  "
          f"peak_vram={m['peak_vram_mib']} MiB")
    print(f'    eval curve: {curve}')

Measured final run:

| adapter | eval curve (25%->100%) | train loss | trainable | time |
|---|---|---|---|---|
| legal | 1.310 -> 1.203 -> 1.165 -> **1.160** | 1.409 | 24.3 M (0.75%) | 18.6 min |
| news | 1.934 -> 1.909 -> 1.904 -> **1.904** | 1.949 | 29.9 M (0.96%) | 28.2 min |
| general_qa | 3.959 -> 2.185 -> 1.666 -> **1.648** | 4.013 | 28.2 M (1.12%) | 44.2 min |

**Note on 'identical budget':** what is held identical is the *budget*, not the
resulting adapter capacity. The same rank yields different trainable counts
across architectures, so the news expert gets ~23% more capacity than the legal
one. That is inherent to specialising three different families and compounds the
model-position entanglement in Threats to Validity.

## 3.3 Loss is not a quality signal here

**Do not conclude an adapter is good from its loss curve.** An earlier adapter
set reduced held-out loss by 46-72% and generated degenerate word-salad in the
deployed graph. Every quality judgement in nb 04 is made against the *base
model* and against *real prompts*, never against training loss.

In [ ]:
# Held-out loss of the UNADAPTED base models, for reference.
# Needed because the first eval point already contains a full epoch of training,
# so an epoch-to-epoch curve is blind to everything learned in epoch 1. Judged
# on that curve alone the news adapter looked inert - yet it had cut loss from
# 2.994 to 1.630 within one epoch and then saturated.
!.venv\Scripts\python.exe finetune\base_eval_loss.py

Measured (earlier 3-epoch adapter set, illustrating the point):

| role | base | best | gain |
|---|---|---|---|
| legal | 2.958 | 0.816 | -72% |
| news | 2.994 | 1.630 | -46% |
| general_qa | 3.449 | 1.096 | -68% |

Large gains - and that set still produced unusable output. Hence nb 04.

## 3.4 Abandoned attempts, for the record

| attempt | settings | outcome |
|---|---|---|
| 1 | 3 ep, lr 2e-4, 512 tok, 2500 ex | 83% of NewsQA targets truncated away; news adapter learned nothing |
| 2 | 3 ep, lr 2e-4, 512 tok, 1500 ex | all three trained, but epoch 3 degraded all of them; answers collapsed in length |
| **3 (final)** | **1 ep, lr 5e-5, 1024 tok, 1500 ex** | **clean monotonic convergence** |

Attempt 2 was discarded and all three retrained rather than patching only the
news adapter, because a mixed budget is the confound described at the top.